# Gaussian elimination with partial pivoting — PyTorch / Kaggle

Attach bcsstk17.mat using Add Input, or set MAT_PATH. Existing outputs below are retained from the earlier Tesla T4 run; they are not new validation results. This is the existing Gaussian Kaggle notebook; its 10^4 default is deliberately preserved.


In [9]:
import time
from pathlib import Path

import numpy as np
import torch
from scipy.io import loadmat
from scipy.sparse import issparse

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64
print("PyTorch:", torch.__version__)
print("Device:", torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU")

PyTorch: 2.11.0+cu130
Device: Tesla T4


In [10]:
@torch.no_grad()
def gaussian_with_partial_pivoting(A, b):
    if A.ndim != 2 or A.shape[0] != A.shape[1] or A.shape[0] == 0:
        raise ValueError("A must be a nonempty square matrix")
    if b.ndim != 1 or b.shape[0] != A.shape[0]:
        raise ValueError("b must be a vector matching A")
    if A.dtype != torch.float64 or b.dtype != torch.float64 or A.device != b.device:
        raise ValueError("A and b must be float64 tensors on the same device")
    if not (torch.isfinite(A).all() and torch.isfinite(b).all()):
        raise ValueError("A and b must be finite")

    LU, c = A.clone().contiguous(), b.clone().contiguous()
    n = len(c)
    for k in range(n):
        r = k + torch.argmax(LU[k:, k].abs()).item()
        if r != k:
            LU[[k, r], :] = LU[[r, k], :]
            c[[k, r]] = c[[r, k]]
        pivot = LU[k, k]
        if not torch.isfinite(pivot).item() or pivot.item() == 0.0:
            raise ValueError(f"Zero or nonfinite pivot at column {k}")
        if k + 1 < n:
            LU[k+1:, k].div_(pivot)
            multipliers = LU[k+1:, k].clone()
            pivot_row = LU[k, k+1:].clone()
            LU[k+1:, k+1:].addr_(multipliers, pivot_row, beta=1, alpha=-1)
            c[k+1:].add_(multipliers * c[k], alpha=-1)
    return LU, c


@torch.no_grad()
def backward_substitution(LU, c):
    x = torch.empty_like(c)
    for i in range(len(c) - 1, -1, -1):
        x[i] = (c[i] - torch.dot(LU[i, i+1:], x[i+1:])) / LU[i, i]
    return x


def solve_ge(A, b):
    LU, c = gaussian_with_partial_pivoting(A, b)
    return backward_substitution(LU, c)


def timed_solve(fn, A, b):
    if A.is_cuda:
        torch.cuda.synchronize(A.device)
    start = time.perf_counter()
    x = fn(A, b)
    if A.is_cuda:
        torch.cuda.synchronize(A.device)
    return x, time.perf_counter() - start


def report(label, A, b, x, x_true, seconds):
    residual = (torch.linalg.vector_norm(A @ x - b) / torch.linalg.vector_norm(b)).item()
    error = (torch.linalg.vector_norm(x - x_true) / torch.linalg.vector_norm(x_true)).item()
    print(f"{label}: {seconds:.6f} s | relative residual {residual:.3e} | relative solution error {error:.3e}")

## 3x3

The repository example has solution `[1, 1, 2]`. The second system forces row swaps at multiple elimination steps. A small warm-up initializes the kernels before timing the SuiteSparse solve.

In [11]:
A_small = torch.tensor([[2, 1, 1], [4, 3, 3], [8, 7, 9]], dtype=dtype, device=device)
b_small = torch.tensor([5, 13, 33], dtype=dtype, device=device)
x_small = solve_ge(A_small, b_small)
torch.testing.assert_close(x_small, torch.tensor([1, 1, 2], dtype=dtype, device=device))
print("Repository example:", x_small.cpu().numpy())

A_swap = torch.tensor([[0, 2, 1], [1, 0, 3], [4, 5, 6]], dtype=dtype, device=device)
x_expected = torch.tensor([1, -2, 3], dtype=dtype, device=device)
b_swap = A_swap @ x_expected
A_before, b_before = A_swap.clone(), b_swap.clone()
torch.testing.assert_close(solve_ge(A_swap, b_swap), x_expected, rtol=1e-12, atol=1e-12)
assert torch.equal(A_swap, A_before) and torch.equal(b_swap, b_before)
print("Row-swap and input-preservation checks passed")

_ = solve_ge(A_small, b_small)
_ = torch.linalg.solve(A_small, b_small)
if device.type == "cuda":
    torch.cuda.synchronize(device)

Repository example: [1. 1. 2.]
Row-swap and input-preservation checks passed


## Obtain the SuiteSparse MATLAB file


In [ ]:
MATRIX = "bcsstk17"
MAT_PATH = None

if MAT_PATH is not None:
    path = Path(MAT_PATH)
    if not path.is_file():
        raise FileNotFoundError(path)
else:
    matches = sorted(Path("/kaggle/input").rglob(f"{MATRIX}.mat"))
    if not matches:
        local = Path("data") / f"{MATRIX}.mat"
        if local.is_file():
            matches = [local]
    if len(matches) != 1:
        raise FileNotFoundError(f"Attach {MATRIX}.mat using Add Input or set MAT_PATH. Matches: {matches}")
    path = matches[0]
print("File:", path)


File: /kaggle/input/datasets/rajathrajesh12/bcsstk17/bcsstk17.mat


In [13]:
mat = loadmat(path)
A_sparse = mat["Problem"][0, 0]["A"]
if A_sparse.shape[0] != A_sparse.shape[1]:
    raise ValueError("Expected a square matrix")
print("Shape:", A_sparse.shape)
print(f"Each dense float64 matrix: {A_sparse.shape[0]**2 * 8 / 2**30:.2f} GiB")
A_numpy = A_sparse.toarray() if issparse(A_sparse) else np.asarray(A_sparse)
A = torch.tensor(A_numpy, dtype=dtype, device=device)
del A_numpy, A_sparse, mat
x_true = torch.ones(A.shape[0], dtype=dtype, device=device)
b = A @ x_true
print("dtype:", A.dtype, "device:", A.device)

Shape: (10974, 10974)
Each dense float64 matrix: 0.90 GiB
dtype: torch.float64 device: cuda:0


## Solve and measure


In [14]:
x, seconds = timed_solve(solve_ge, A, b)
report("Gaussian elimination", A, b, x, x_true, seconds)

Gaussian elimination: 35.474346 s | relative residual 2.391e-15 | relative solution error 9.679e-13
